## Importing necessary libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

## Import Dataset

In [ ]:
df=pd.read_csv('./drive/MyDrive/data/fashion.csv')

# EDA

In [ ]:
df.head()

In [ ]:
df.info()
df['Colour'].value_counts()

The `TripletDataset` expects a specific directory structure with `anchor`, `positive`, and `negative` subdirectories. The previous error indicated that these directories were not found. I will create the necessary empty directory structure here.

In [ ]:
import os

def create_triplet_dirs(base_path):
    """Creates the required anchor, positive, and negative subdirectories."""
    for sub_dir_name in ['anchor', 'positive', 'negative']:
        path = os.path.join(base_path, sub_dir_name)
        os.makedirs(path, exist_ok=True)
        print(f"Created directory: {path}")

# Define the base paths for training and testing data
train_base_path = './drive/MyDrive/data/Apparel/Boys/'
test_base_path = './drive/MyDrive/data/Apparel/Boys/'

print("Creating directories for training data...")
create_triplet_dirs(train_base_path)

print("\nCreating directories for testing data...")
create_triplet_dirs(test_base_path)

print("\nDirectory structure created.")

### Generating Triplet Dataset Structure

To ensure the `TripletDataset` can find the necessary images, we need to populate the `anchor`, `positive`, and `negative` subdirectories. The `generate_triplets` function below will do this by:

1.  **Defining Similarity**: For an anchor image, a 'positive' image will share the same `Category`, `SubCategory`, and `Colour` (based on your criteria).
2.  **Defining Dissimilarity**: A 'negative' image will have at least one different attribute (Category, SubCategory, or Colour) compared to the anchor.
3.  **Copying Images**: It will copy the relevant images from a specified `original_image_folder` into the `anchor`, `positive`, and `negative` directories.

In [ ]:
class L2Normalize(nn.Module):
    def __init__(self, p=2, dim=1, eps=1e-12):
        super().__init__()
        self.p = p
        self.dim = dim
        self.eps = eps

    def forward(self, x):
        return nn.functional.normalize(x, p=self.p, dim=self.dim, eps=self.eps)

In [ ]:
def get_resnet50_encoder(embedding_dim=512, pretrained=True, train_backbone=False):
    resnet = models.resnet50(pretrained=pretrained)
    modules = list(resnet.children())[:-1]  # Remove last FC layer
    backbone = nn.Sequential(*modules)

    model = nn.Sequential(
        backbone,
        nn.Flatten(),
        nn.Linear(2048, embedding_dim),
        L2Normalize()  # L2 normalize embeddings
    )

    # Optionally freeze backbone
    if not train_backbone:
        for param in backbone.parameters():
            param.requires_grad = False

    return model

In [ ]:
# ===== 2. Dataset & DataLoader =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
import os
from PIL import Image
import random
from torch.utils.data import Dataset
from torchvision import transforms

class TripletDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.anchor_dir = os.path.join(root_dir, 'anchor')
        self.positive_dir = os.path.join(root_dir, 'positive')
        self.negative_dir = os.path.join(root_dir, 'negative')

        if not os.path.isdir(self.anchor_dir) or \
           not os.path.isdir(self.positive_dir) or \
           not os.path.isdir(self.negative_dir):
            raise RuntimeError(f"Expected 'anchor', 'positive', and 'negative' subdirectories in {root_dir}")

        self.triplets = self._find_triplets()

        if not self.triplets:
            raise RuntimeError(f"Found 0 valid triplets in {root_dir}. Ensure corresponding images exist in anchor, positive, and negative subdirectories.")

    def _find_triplets(self):
        triplets = []
        anchor_images = sorted([f for f in os.listdir(self.anchor_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))])

        for img_name in anchor_images:
            positive_path = os.path.join(self.positive_dir, img_name)
            negative_path = os.path.join(self.negative_dir, img_name)

            if os.path.exists(positive_path) and os.path.exists(negative_path):
                triplets.append((os.path.join(self.anchor_dir, img_name), positive_path, negative_path))
        return triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        anchor_path, positive_path, negative_path = self.triplets[idx]

        anchor_img = Image.open(anchor_path).convert("RGB")
        positive_img = Image.open(positive_path).convert("RGB")
        negative_img = Image.open(negative_path).convert("RGB")

        if self.transform:
            anchor_img = self.transform(anchor_img)
            positive_img = self.transform(positive_img)
            negative_img = self.transform(negative_img)

        return anchor_img, positive_img, negative_img

In [ ]:
# Update the dataset and dataloader creation with the new TripletDataset

try:
    # Use the defined train_base_path and test_base_path for TripletDataset
    train_dataset = TripletDataset(train_base_path, transform=transform)
    val_dataset = TripletDataset(test_base_path, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./drive/MyDrive/data/Apparel/Boys/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./drive/MyDrive/data/Apparel/Boys/anchor/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/positive/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/negative/image1.jpg")

In [ ]:
import os
import shutil

# Path to the .ipynb_checkpoints directory that might be causing issues
checkpoint_dir_train = os.path.join(train_base_path, '.ipynb_checkpoints')
checkpoint_dir_val = os.path.join(test_base_path, '.ipynb_checkpoints')

# Check if the directory exists and remove it
if os.path.exists(checkpoint_dir_train) and os.path.isdir(checkpoint_dir_train):
    shutil.rmtree(checkpoint_dir_train)
    print(f"Removed problematic directory: {checkpoint_dir_train}")

if os.path.exists(checkpoint_dir_val) and os.path.isdir(checkpoint_dir_val):
    shutil.rmtree(checkpoint_dir_val)
    print(f"Removed problematic directory: {checkpoint_dir_val}")

try:
    # Use the defined train_base_path and test_base_path for TripletDataset
    train_dataset = TripletDataset(train_base_path, transform=transform)
    val_dataset = TripletDataset(test_base_path, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
    print("Triplet Datasets and DataLoaders created successfully!")
except RuntimeError as e:
    print(f"Error creating Triplet Dataset: {e}")
    print("Please ensure your root directories (e.g., ./drive/MyDrive/data/Apparel/Boys/) contain 'anchor', 'positive', and 'negative' subdirectories, and that corresponding image files exist in each.")
    print("Example: ./drive/MyDrive/data/Apparel/Boys/anchor/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/positive/image1.jpg, ./drive/MyDrive/data/Apparel/Boys/negative/image1.jpg")

# ===== 3. Model, Loss, Optimizer =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_resnet50_encoder(embedding_dim=256, pretrained=True, train_backbone=True).to(device)

# Example: contrastive learning loss (InfoNCE, Triplet, etc.)
# Here we use TripletMarginLoss for demonstration
criterion = nn.TripletMarginLoss(margin=1.0, p=2)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc="Training"):
        # For Triplet loss, you need (anchor, positive, negative) samples
        # Here we assume you have a custom dataset that returns them
        # This is just a placeholder
        anchor, positive, negative = batch  # Replace with your triplet dataset
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

        optimizer.zero_grad()
        emb_a = model(anchor)
        emb_p = model(positive)
        emb_n = model(negative)

        loss = criterion(emb_a, emb_p, emb_n)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validating"):
            anchor, positive, negative = batch
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)

            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)

            loss = criterion(emb_a, emb_p, emb_n)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 10
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = validate_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")

# ===== 7. Save the trained encoder =====
torch.save(model.state_dict(), "resnet50_encoder.pth")
print("Model saved as resnet50_encoder.pth")

In [ ]:
import matplotlib.pyplot as plt

# Plotting the training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_losses, label='Training Loss')
plt.plot(range(1, EPOCHS + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()
